In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("🚀 Starting Step 3: Data Preprocessing & Feature Engineering...")

# ==========================================
# 1. LOAD THE RAW DATA
# ==========================================
# Load the CSV generated in the previous step
try:
    df = pd.read_csv('esp32_solar_energy_dataset.csv')
    print(f"✅ Loaded dataset with {len(df)} rows.")
except FileNotFoundError:
    print("❌ Error: 'esp32_solar_energy_dataset.csv' not found. Please run the data generation script first.")
    exit()

# ==========================================
# 2. FEATURE ENGINEERING (The "AI Translation")
# ==========================================
print("🛠️ Engineering new features based on physical behaviors...")

# A. Time-Based Features (Thermodynamic/Environmental context)
# Solar charging only happens during the day. This helps the AI understand battery context.
df['Is_Daytime'] = ((df['Hour_of_Day'] >= 8) & (df['Hour_of_Day'] <= 17)).astype(int)

# B. Delta Features (Rate of Change) - CRUCIAL for detecting physical switching events
# If a relay turns ON, current changes instantly. 'diff()' calculates the step-by-step change.
df['Power_Delta'] = df['Power'].diff().fillna(0)
df['Current_Delta'] = df['Current'].diff().fillna(0)
df['Voltage_Delta'] = df['Voltage'].diff().fillna(0)

# C. Rolling/Statistical Features (Noise Smoothing & Trend Analysis)
# We use a window of 5 steps. This helps the AI see the "trend" rather than reacting to a single noisy spike.
window_size = 5
df['Power_Rolling_Mean'] = df['Power'].rolling(window=window_size, min_periods=1).mean()
df['Current_Rolling_Std'] = df['Current'].rolling(window=window_size, min_periods=1).std().fillna(0) # Volatility

# ==========================================
# 3. DATA CLEANING & STRATEGIC SPLITTING
# ==========================================
print("🧹 Cleaning data and creating specialized datasets...")

# Create a "Clean" dataset for Load ID and Prediction (Remove injected anomalies)
df_clean = df[df['Is_Anomaly'] == 0].copy()
print(f"📉 Cleaned dataset size: {len(df_clean)} rows (Removed {len(df) - len(df_clean)} anomalies).")

# ==========================================
# 4. DEFINE FEATURES (X) AND TARGETS (y)
# ==========================================
# These are the inputs the AI will look at. We use our newly engineered features!
feature_cols = [
    'Voltage', 'Current', 'Power', 'Relay_1', 'Relay_2', 'Battery_Level',
    'Hour_of_Day', 'Is_Daytime',       # Context features
    'Power_Delta', 'Current_Delta', 'Voltage_Delta', # Physical switching features
    'Power_Rolling_Mean', 'Current_Rolling_Std'      # Noise/Trend features
]

# --- DATASET A: For Load Identification & Power Prediction (Clean Data Only) ---
X_clean = df_clean[feature_cols]
y_load_id = df_clean['Load_Type']             # Target: What appliance is running?
y_pred_power = df_clean['Predicted_Power_Next'] # Target: What will power be next step?

# --- DATASET B: For Anomaly Detection (Full Data, INCLUDING Faults) ---
X_full = df[feature_cols]
y_anomaly = df['Is_Anomaly']                  # Target: Is this a fault? (0 or 1)

# ==========================================
# 5. TRAIN / TEST SPLITTING
# ==========================================
print("🔀 Splitting data into Training (80%) and Testing (20%) sets...")

# Split Clean Data (For Load ID & Prediction)
# We use 'stratify' to ensure the test set has the same proportion of each Load Type as the training set.
X_clean_train, X_clean_test, y_load_train, y_load_test = train_test_split(
    X_clean, y_load_id, test_size=0.2, random_state=42, stratify=y_load_id
)

X_clean_train_pred, X_clean_test_pred, y_pred_train, y_pred_test = train_test_split(
    X_clean, y_pred_power, test_size=0.2, random_state=42
)

# Split Full Data (For Anomaly Detection)
# Note: Anomalies are rare, so stratify is very important here to ensure we have faults in both train and test.
X_full_train, X_full_test, y_anom_train, y_anom_test = train_test_split(
    X_full, y_anomaly, test_size=0.2, random_state=42, stratify=y_anomaly
)

# ==========================================
# 6. FEATURE SCALING (Normalization)
# ==========================================
print("📏 Scaling features (Voltage is ~230, Current is ~5, Battery is ~100)...")

# AI models struggle if features have vastly different scales. We normalize them to have a mean of 0 and variance of 1.
scaler_clean = StandardScaler()
X_clean_train_scaled = scaler_clean.fit_transform(X_clean_train)
X_clean_test_scaled = scaler_clean.transform(X_clean_test) # Note: Only transform test data!

scaler_full = StandardScaler()
X_full_train_scaled = scaler_full.fit_transform(X_full_train)
X_full_test_scaled = scaler_full.transform(X_full_test)

print("\n✅ SUCCESS! Data Preprocessing and Feature Engineering is complete.")
print("📊 Your data is now perfectly formatted and ready for Machine Learning models.")
csv_filename = "esp32_solar_energy_dataset_ehanced.csv"
df.to_csv(csv_filename, index=False)
print(f"\n✅ Dataset successfully generated and saved as '{csv_filename}'!")


🚀 Starting Step 3: Data Preprocessing & Feature Engineering...
✅ Loaded dataset with 5000 rows.
🛠️ Engineering new features based on physical behaviors...
🧹 Cleaning data and creating specialized datasets...
📉 Cleaned dataset size: 4950 rows (Removed 50 anomalies).
🔀 Splitting data into Training (80%) and Testing (20%) sets...
📏 Scaling features (Voltage is ~230, Current is ~5, Battery is ~100)...

✅ SUCCESS! Data Preprocessing and Feature Engineering is complete.
📊 Your data is now perfectly formatted and ready for Machine Learning models.

✅ Dataset successfully generated and saved as 'esp32_solar_energy_dataset_ehanced.csv'!
